# Statistical Foundations & Predictive Maintenance Lab
This notebook performs cleaning, hypothesis testing, feature engineering, modeling, and RUL estimation on the synthetic PdM dataset.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, mean_absolute_error
from sklearn.linear_model import LinearRegression

DATA_PATH = '../data/pdm_synthetic_sensor_data.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df.shape, df.head()

In [ ]:
# Descriptive statistics and cleaning checks
numeric = df.select_dtypes(include='number')
summary = numeric.describe(percentiles=[.05, .25, .5, .75, .95]).T
constant_columns = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
missing_counts = df.isna().sum().sort_values(ascending=False)
summary.head(), constant_columns, missing_counts.head()

In [ ]:
# Three-standard-deviation outlier flags
z = (numeric - numeric.mean()) / numeric.std(ddof=0)
outlier_counts = (z.abs() > 3).sum().sort_values(ascending=False)
outlier_counts.head(10)

In [ ]:
# Independent t-test: Day vs Night vibration levels
day = df.loc[df['shift'] == 'Day', 'vibration_g'].dropna()
night = df.loc[df['shift'] == 'Night', 'vibration_g'].dropna()
t_stat, p_value = stats.ttest_ind(day, night, equal_var=False)
print({'t_statistic': round(t_stat, 4), 'p_value': round(p_value, 6), 'day_mean': day.mean(), 'night_mean': night.mean()})
print('Reject H0: shift means differ' if p_value < 0.05 else 'Fail to reject H0')

In [ ]:
# ANOVA: throughput proxy across depots using pressure
samples = [g['pressure_psi'].dropna() for _, g in df.groupby('depot')]
f_stat, anova_p = stats.f_oneway(*samples)
print({'f_statistic': round(f_stat, 4), 'p_value': round(anova_p, 6)})

In [ ]:
# Feature engineering: rolling mean, RMS, and peak-to-peak
work = df.sort_values(['asset_id', 'timestamp']).copy()
work['vibration_roll_mean_6'] = work.groupby('asset_id')['vibration_g'].transform(lambda s: s.rolling(6, min_periods=1).mean())
work['vibration_rms_6'] = work.groupby('asset_id')['vibration_g'].transform(lambda s: np.sqrt(s.pow(2).rolling(6, min_periods=1).mean()))
work['vibration_peak_to_peak_6'] = work.groupby('asset_id')['vibration_g'].transform(lambda s: s.rolling(6, min_periods=1).max() - s.rolling(6, min_periods=1).min())
work[['asset_id','timestamp','vibration_g','vibration_roll_mean_6','vibration_rms_6','vibration_peak_to_peak_6']].head(10)

In [ ]:
# Bagged tree classifier: predict failure within 7 days
features = ['asset_age_months','vibration_g','temperature_c','pressure_psi','acoustic_db','motor_current_a','error_events','vibration_roll_mean_6','vibration_rms_6','vibration_peak_to_peak_6']
model_df = work[features + ['failure_within_7_days']].dropna()
X_train, X_test, y_train, y_test = train_test_split(model_df[features], model_df['failure_within_7_days'], test_size=0.25, random_state=42, stratify=model_df['failure_within_7_days'])
clf = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5, random_state=42), n_estimators=40, random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=['No near failure','Near failure']))

In [ ]:
# Remaining Useful Life regression baseline
reg = LinearRegression()
rul_df = work[features + ['time_to_failure_hours']].dropna()
X_train, X_test, y_train, y_test = train_test_split(rul_df[features], rul_df['time_to_failure_hours'], test_size=0.25, random_state=42)
reg.fit(X_train, y_train)
rul_pred = reg.predict(X_test)
print({'mae_hours': round(mean_absolute_error(y_test, rul_pred), 2)})